## Notebook for testing our pipeline for the Generative Recommender

In [1]:
# Imports here
import time
import torch
import numpy as np
import pickle
import os, sys
import joblib
import torch, numpy as np, random
import pandas as pd
from datetime import datetime


project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.data_utils.data_handler as dh
from src.components.embedder.embedder import Embedder
from src.components.quantizer.rqvae import RQVAE
from train.rqvae_train import rqvae_loss, train_rqvae_sanity_check, train_rqvae_full, load_trained_rqvae
from src.train.train_transformer import start_training
from src.components.transformer.transformer import recommended_next_sid, is_model_trained
from transformers import BartTokenizer, BartForConditionalGeneration, BartTokenizerFast

from src.eval.baseline.collaborative_filtering import compute_item_user_matrix, recommend_for_user
from src.eval.evaluation import MAP_at_K



# This is ONLY for our tests! Do not use for our full model
'''
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False'''

'\ndef set_seed(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False'

### First, get the feature strings from DataHandler

In [2]:
'''import os
import transformers
path = './../models/bart-recommender_iteration2/final_model'
print(os.listdir(path))

print(transformers.__version__)

tok_fast = BartTokenizerFast.from_pretrained('./../models/bart-recommender_iteration2/final_model')
print("Fast vocab:", len(tok_fast))

tok_slow = BartTokenizer.from_pretrained('./../models/bart-recommender_iteration2/final_model')
print("Slow vocab:", len(tok_slow))'''

feature_strings = dh.get_article_feature_string_list()
# feature_strings is a list of lists, use list comprehension to make it a list of strings (first 1000 items)
feature_strings = [item[0] for item in feature_strings]
item_ids = [feature_string.split()[0] for feature_string in feature_strings]

print(f'Number of items: {len(feature_strings)}')
print(feature_strings[0])


C:\Users\motes\KTH\KTH 2025 HT\DD2430 Data Science Project course\generative-recommendation\src
Number of items: 105542
108775015 Solid Dark Black Strap top Jersey top with narrow shoulder straps. Vest top Garment Upper body Jersey Basic Ladieswear Ladieswear Womens Everyday Basics Jersey Basic


### Next, we use the embedder to get the embeddings that will serve as input for RQ-VAE

In [2]:
if os.path.exists('./../data/embeddings/bert-large_embeddings.npy'):
    embeddings = np.load('./../data/embeddings/bert-large_embeddings.npy')
    print(f'The embeddings have the shape: {embeddings.shape}')
else:
    embedder = Embedder("sbert")
    embeddings = embedder.encode(feature_strings) # embeddings has shape (n, d), where d = 384 for SBERT
    print(f'The embeddings have the shape: {embeddings.shape}')
    # print(embeddings[0])
    np.save('./../data/embeddings/bert-large_embeddings.npy', embeddings)

The embeddings have the shape: (105542, 1024)


### Now, we can use the embeddings with RQ-VAE. The code loads hashmaps between item IDs and semantic IDs if they exist. If not, it uses the model to generate semantic IDs and creates (and saves) the hashmaps

In [3]:
input_dim = embeddings.shape[1]
latent_dim = 64
hidden_dim = 512
codebook_size = 512
num_quantizers = 5

TRAINING_MODE = 'None' # for loading

#set_seed(42)
rqvae = RQVAE(input_dim=input_dim, latent_dim=latent_dim, hidden_dim=hidden_dim, codebook_size=512, num_quantizers=num_quantizers)

# We need to train the model here, before retrieving semantic IDs! Omitting for now... 
# trained_model = ...

print('Generating semantic IDs')

if isinstance(embeddings, np.ndarray):
    embeddings = torch.from_numpy(embeddings).float()

if TRAINING_MODE == 'sanity_check':
    print("Running sanity check for RQ-VAE")
    rqvae = train_rqvae_sanity_check(rqvae, embeddings)

elif TRAINING_MODE == 'full_training':
    print("Training RQ-VAE")
    rqvae = train_rqvae_full(rqvae, embeddings, save_path='./../models/rqvae_iteration2/final_models')

else:
    rqvae = load_trained_rqvae(rqvae, './../models/rqvae_iteration2/final_models/rqvae_best.pth')

semantic_ids = rqvae.encode_to_semantic_ids(embeddings)
semantic_ids = semantic_ids.detach().numpy() # List of semantic IDs

print(f'semantic_ids has shape: {semantic_ids.shape}')
print('printing the first semantic ID (trained RQ-VAE)')
print(semantic_ids[0])

item_2_semantic = {}
semantic_2_item = {}

if os.path.exists('./../data/semantic_ids/item_2_semantic_20251127_132421.pkl') and os.path.exists('./../data/semantic_ids/semantic_2_item_20251127_132421.pkl'):
    print('Loading existing hashmaps')

    with open('./../data/semantic_ids/item_2_semantic_20251127_132421.pkl', 'rb') as f:
        item_2_semantic = pickle.load(f)
    with open('./../data/semantic_ids/semantic_2_item_20251127_132421.pkl', 'rb') as f:
        semantic_2_item = pickle.load(f)
    print('Loaded the hashmaps')

else:
    for item_id, semantic_id in zip(item_ids, semantic_ids):
        semantic_tuple = tuple(semantic_id.astype(int))
        item_2_semantic[item_id] = semantic_tuple
        semantic_2_item[semantic_tuple] = item_id

    print(f'Done creating hashmaps. Saving...')
    with open('./../data/semantic_ids/item_2_semantic_20251127_132421.pkl', 'wb') as f:
        pickle.dump(item_2_semantic, f)

    with open('./../data/semantic_ids/semantic_2_item_20251127_132421.pkl', 'wb') as f:
        pickle.dump(semantic_2_item, f)
    print(f'Saved hashmaps to files.')

Generating semantic IDs
semantic_ids has shape: (105542, 5)
printing the first semantic ID (trained RQ-VAE)
[  2  27 315  31 175]
Loading existing hashmaps
Loaded the hashmaps


### Converting transactions data into: 
#### i.e.: user_id: item_id1, ..., item_idn --> user_id: item_SID1, ..., item_SIDn

In [4]:
transactions_train = pd.read_pickle('./../data/transaction_list_train.pkl')
transactions_val = pd.read_pickle('./../data/transaction_list_val.pkl')
transactions_test = pd.read_pickle('./../data/transaction_list_test.pkl')

customer_transactions_train = {}
customer_transactions_val = {}
customer_transactions_test = {}

if os.path.exists('./../data/customer_transactions_train.pkl') and os.path.exists('./../data/customer_transactions_val.pkl') and os.path.exists('./../data/customer_transactions_test.pkl'):
    print('Loading existing customer_transactions_train and customer_transactions_val')

    with open('./../data/customer_transactions_train.pkl', 'rb') as f:
        customer_transactions_train = pickle.load(f)
    with open('./../data/customer_transactions_val.pkl', 'rb') as f:
        customer_transactions_val = pickle.load(f)
    with open('./../data/customer_transactions_test.pkl', 'rb') as f:
        customer_transactions_test = pickle.load(f)

    print('Loaded existing customer_transactions_train and customer_transactions_val')
else:

    for customer_id, group in transactions_train.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_train[customer_id] = sid_list
    
    for customer_id, group in transactions_val.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_val[customer_id] = sid_list
    
    for customer_id, group in transactions_test.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_test[customer_id] = sid_list
        
    print("customer_transactions_train, customer_transactions_val and customer_transactions_test have been created, saving...")
    with open("./../data/customer_transactions_train.pkl", "wb") as f:
        pickle.dump(customer_transactions_train, f)
    with open('./../data/customer_transactions_val.pkl', 'wb') as f:
        pickle.dump(customer_transactions_val, f)
    with open('./../data/customer_transactions_test.pkl', 'wb') as f:
        pickle.dump(customer_transactions_test, f)
    print("Saving done!")

# sanity check
first_customer = list(customer_transactions_test.keys())[0]
print(f"Customer_id: {first_customer}")
print(f"Train sequence => length: {len(customer_transactions_train[first_customer])} : {customer_transactions_train[first_customer]}")
print(f"Val sequence => length: {len(customer_transactions_val[first_customer])} : {customer_transactions_val[first_customer]}")
print(f"Test sequence => length: {len(customer_transactions_test[first_customer])} : {customer_transactions_test[first_customer]}")

Loading existing customer_transactions_train and customer_transactions_val
Loaded existing customer_transactions_train and customer_transactions_val
Customer_id: 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657
Train sequence => length: 19 : [(np.int64(139), np.int64(283), np.int64(20), np.int64(354)), (np.int64(462), np.int64(79), np.int64(375), np.int64(252)), (np.int64(288), np.int64(108), np.int64(30), np.int64(348)), (np.int64(218), np.int64(202), np.int64(120), np.int64(403)), (np.int64(502), np.int64(423), np.int64(113), np.int64(70)), (np.int64(502), np.int64(423), np.int64(113), np.int64(70)), (np.int64(321), np.int64(423), np.int64(330), np.int64(94)), (np.int64(381), np.int64(122), np.int64(356), np.int64(231)), (np.int64(260), np.int64(261), np.int64(303), np.int64(53)), (np.int64(502), np.int64(219), np.int64(312), np.int64(310)), (np.int64(502), np.int64(219), np.int64(312), np.int64(310)), (np.int64(303), np.int64(219), np.int64(120), np.int64(30)), (np.i

### Training the transformer

In [ ]:
if is_model_trained(): 
    print('The model is loaded...')
    model = BartForConditionalGeneration.from_pretrained('./../models/bart-recommender_iteration2/final_model')
    tokenizer = BartTokenizer.from_pretrained('./../models/bart-recommender_iteration2/final_model')
else: 
    print('There is no pretrained model, the model will be trained ...')
    start_training() # train the model

There is no pretrained model, the model will be trained ...
Loading existing customer_transactions_train and customer_transactions_val
The files are loaded!
Model is training ...


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
C:\Users\motes\anaconda3\envs\gr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


### Retrieving articles info for mapping

In [7]:
articles = pd.read_pickle('./../data/articles.pkl')
article_id = 108775044
row = articles.loc[articles['article_id'] == article_id]
product_name = row['prod_name']
product_name2 = row['prod_name'].iloc[0]
print(product_name)
print(product_name2)

1    Strap top
Name: prod_name, dtype: object
Strap top


### Inference: 

### Sanity check: GR recommends SIDs for a given customer transaction sequence

In [10]:
last_10_customers = list(customer_transactions_val.keys())[-1:]
test_customers_sequences = {k: customer_transactions_val[k] for k in last_10_customers}

for customer_id, seqs in test_customers_sequences.items():
    test_seq = test_customers_sequences[customer_id]
    test_seq = [' '.join(tuple(str(x) for x in sid)) for sid in test_seq]
    print(f"Customer: {customer_id}")
    print(f"Input SIDs sequence: {test_seq}")
    recommended_sids = recommended_next_sid(test_seq, model, tokenizer, window_size=36, top_k=12)
    print(recommended_sids)
    filtered = []
    for sid in recommended_sids: # to filter non-existing keys
        key = tuple(sid)
        if not key:
            continue
        item = semantic_2_item.get(key)
        if item is None:
            continue
        filtered.append((sid, item))
    
    print("Recommended SIDs, generated by transformer: ")
    for sid, id in filtered:
        row = articles.loc[articles['article_id'].astype(str) == str(id)]
        prod_name = row['prod_name'].iloc[0]
        print(f'Rec. SID: {sid} --> article_id: {id} --> product_name: {prod_name}')

Customer: ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38b2236865d949d4df6a
Input SIDs sequence: ['311 467 460 249', '501 498 64 500', '358 486 97 213', '148 391 86 189', '466 408 32 167', '148 391 86 189']
[[169, 422, 231, 72, 242, 0], [304, 299, 218, 250, 316, 0], [304, 387, 317, 447, 120, 0], [169, 56, 71, 213, 242, 0], [304, 199, 450, 34, 153, 0], [304, 199, 450, 34, 229, 0], [169, 393, 421, 280, 242, 1], [169, 56, 231, 213, 242, 0], [169, 56, 287, 6, 108, 0], [304, 199, 450, 199, 229, 0], [304, 199, 450, 199, 153, 0], [169, 393, 421, 280, 242, 0]]
Recommended SIDs, generated by transformer: 
Rec. SID: [169, 422, 231, 72, 242, 0] --> article_id: 720125001 --> product_name: SUPREME RW tights
Rec. SID: [304, 299, 218, 250, 316, 0] --> article_id: 610776002 --> product_name: Tilly (1)
Rec. SID: [304, 387, 317, 447, 120, 0] --> article_id: 610776001 --> product_name: Tilly (1)
Rec. SID: [169, 56, 71, 213, 242, 0] --> article_id: 720125007 --> product_name: SUPREME RW tights
Rec. SID: [3

# Evaluation Code Generative Recommendation (1% of test dataset)

In [ ]:
model = BartForConditionalGeneration.from_pretrained('./../models/bart-recommender_iteration2/final_model')
tokenizer = BartTokenizer.from_pretrained('./../models/bart-recommender_iteration2/final_model')

transaction_list_eval = pd.read_pickle("./../data/customer_transactions_test.pkl") # evaluation on test dataset
items = list(transaction_list_eval.values())
fraction_01p = int(len(items) * 0.01)
transaction_list_eval_sample = items[:fraction_01p]

# Have a U x 1 np.array with labels
lab = [lst[-1] for (lst) in transaction_list_eval_sample]
lab = [' '.join(tuple(str(x) for x in sid)) for sid in lab]

# Recommend K products to U customers
start = time.time()
pred = []
for seq in transaction_list_eval_sample:
    seq_norm = [' '.join(map(str, sid)) for sid in seq]
    rec = recommended_next_sid(seq_norm, model, tokenizer, window_size=37, top_k=12)
    rec_str = [' '.join(map(str, sid)) for sid in rec]
    pred.append(rec_str)
elapsed_time = time.time() - start


# Evaluate using both pred and lab
map12 = MAP_at_K(pred, lab)
print(f"MAP@12 for GR is {map12}")
print(f"Total elapsed time for input size {len(transaction_list_eval_sample)}: {elapsed_time}")

# Evaluation Code Collaborative filtering (1% of test dataset)

In [6]:
if os.path.exists('./../data/CF_cashe.joblib'):
    unique_items, user2idx, interaction_matrix, item_similarity = joblib.load('./../data/CF_cashe.joblib')
else:
    unique_items, user2idx, interaction_matrix, item_similarity = compute_item_user_matrix()
    joblib.dump((unique_items,user2idx, interaction_matrix, item_similarity), "./../data/CF_cashe.joblib")

transaction_list_eval = pd.read_pickle("transaction_list_test.pkl")
fraction_01p = int(len(transaction_list_eval)*0.01)
transaction_list_eval01p = transaction_list_eval[:fraction_01p]
lab = [lst[-1] for lst in transaction_list_eval01p.iloc[:, 1]]
start = time.time()
pred = [recommend_for_user(customer_id, unique_items, user2idx, interaction_matrix, item_similarity, top_k=12)
             for customer_id, _ in transaction_list_eval01p.values] 
elapsed_time = time.time() - start 

map12 = MAP_at_K(pred, lab)
print(f"MAP@12 for CF is {map12}")
print(f"Total elapsed time for input size {len(transaction_list_eval01p)}: {elapsed_time}")

MAP@12 for CF is 0.03531776768111805
Total elapsed time for input size 7629: 7.8684892654418945
